In [1]:
!pip install transformers
!pip install PyMuPDF
!pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 89.9 MB/s eta 0:00:00


In [37]:
import fitz  # PyMuPDF
from transformers import pipeline

In [38]:

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [39]:
import os

drive_folder = ""

drive_path = os.path.join('/content/drive/MyDrive', drive_folder)


if os.path.exists(drive_path):

    pass
else:
    print(f"Folder not found: {drive_path}")


pdf_path = '/content/drive/MyDrive/Test/Test PDF (Hard).pdf'


In [40]:
def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""

    for page in doc:
        text += page.get_text()

    return text

pdf_text = extract_text_from_pdf(pdf_path)

print(pdf_text[:1000])  # preview

Bangladesh(_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&
&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&
&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%
%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%
%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$
$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((
((((((_. is a beautiful country in South Asia, known for its rich cultural heritage, fertile land, and 
vast river systems. The capital city of Bangladesh is Dhaka. Dhaka is not only the political 
center but also the economic and cultural heart of the nation. It is one of the most densely 
populated cities in the world and plays a major role in the country’s development. 
 
One of the most important natural features of Bangladesh is the 
(_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&()
)))))))))(((((((((((((((_$$$$$$$$$$$$$$$

In [44]:
import re

def clean_text(text):
    print("Starting text cleaning...")


    print("  Converting text to lowercase...")
    text = text.lower()


    print("  Removing special characters...")
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)


    print("  Standardizing whitespace...")
    text = re.sub(r'\s+', ' ', text)

    print("  Stripping leading and trailing whitespace...")
    text = text.strip()

    print("Text cleaning complete.")
    return text

pdf_text = clean_text(pdf_text)

print("\nPreview of cleaned text (first 500 characters):")
print(pdf_text[:500])

Starting text cleaning...
  Converting text to lowercase...
  Removing special characters...
  Standardizing whitespace...
  Stripping leading and trailing whitespace...
Text cleaning complete.

Preview of cleaned text (first 500 characters):
bangladesh is a beautiful country in south asia known for its rich cultural heritage fertile land and vast river systems the capital city of bangladesh is dhaka dhaka is not only the political center but also the economic and cultural heart of the nation it is one of the most densely populated cities in the world and plays a major role in the country s development one of the most important natural features of bangladesh is the padma river which is often referred to as the lifeline of the country


In [45]:
def chunk_text(text, chunk_size=400):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunks.append(" ".join(words[i:i+chunk_size]))

    return chunks

chunks = chunk_text(pdf_text)

print("Total chunks:", len(chunks))

Total chunks: 1


In [47]:
qa_pipeline = pipeline(
    "question-answering",
    model="deepset/xlm-roberta-large-squad2",
    tokenizer="deepset/xlm-roberta-large-squad2"
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaForQuestionAnswering LOAD REPORT from: deepset/xlm-roberta-large-squad2
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [48]:
import re

def split_questions(text):
    questions = re.split(r'\?\s*', text)
    questions = [q.strip() + '?' for q in questions if q.strip()]
    return questions

In [62]:
def get_answer(question, chunks):
    best_answer = ""
    best_score = 0

    # print(f"  Searching for answer for: '{question}'...") # Optional: uncomment for verbose searching

    for chunk in chunks:
        result = qa_pipeline(question=question, context=chunk)

        if result['score'] > best_score:
            best_score = result['score']
            best_answer = result['answer']

    # High confidence
    if best_score >= 0.4:
        return best_answer, best_score
    # Medium confidence
    elif best_score >= 0.2:
        return (
            f"Possible Answer (Confidence: {best_score:.2f}): {best_answer}. The model found a potential answer but with moderate confidence. This might be incomplete or need further review.",
            best_score
        )
    # Low confidence
    else:
        return (
            "No Confident Answer Found (Confidence: {best_score:.2f}): The model could not find a clear and confident answer in the document for your question. Please try rephrasing or check if the information is present in the text.",
            best_score
        )

In [63]:
while True:
    user_input = input("\nEnter your question(s) about the PDF (or type 'exit' to quit): ")

    if user_input.lower() == "exit":
        print("\nExiting Q&A session. Goodbye!")
        break

    print("\n" + "="*70)
    print(f"QUERY RECEIVED: {user_input}")
    print("="*70)

    questions = split_questions(user_input)

    if len(questions) > 1:
        print("\nMultiple questions detected. Processing each one individually...")
        for i, question in enumerate(questions, 1):
            print("\n" + "-"*70)
            print(f"Question {i}/{len(questions)}: {question}")
            answer, score = get_answer(question, chunks)
            print(f"Answer: {answer}")
            print(f"Confidence: {score:.2f}")
            print("-"*70)
    else:
        print("\nProcessing single question...")
        print("\n" + "-"*70)
        print(f"Question: {questions[0]}")
        answer, score = get_answer(questions[0], chunks)
        print(f"Answer: {answer}")
        print(f"Confidence: {score:.2f}")
        print("-"*70)

    print("\n" + "="*70 + "\n")


Enter your question(s) about the PDF (or type 'exit' to quit): What is the name of the world’s largest mangrove forest located in Bangladesh?

QUERY RECEIVED: What is the name of the world’s largest mangrove forest located in Bangladesh?

Processing single question...

----------------------------------------------------------------------
Question: What is the name of the world’s largest mangrove forest located in Bangladesh?
Answer: sundarbans
Confidence: 1.67
----------------------------------------------------------------------



Enter your question(s) about the PDF (or type 'exit' to quit): Which currency is used in Bangladesh? What is the national animal of Bangladesh?

QUERY RECEIVED: Which currency is used in Bangladesh? What is the national animal of Bangladesh?

Multiple questions detected. Processing each one individually...

----------------------------------------------------------------------
Question 1/2: Which currency is used in Bangladesh?
Answer: bangladeshi taka
Co